# 02 Clean

Turns the raw Vektis and CBS files into clean tables. Every cleaning decision is written down: what I found, what I did, why.

In [ ]:
# pandas: the library for working with tables
import pandas as pd

In [ ]:
# Load the raw Vektis file (semicolon-separated). The raw file itself is never changed.
vektis = pd.read_csv("../data/raw/Vektis Open Databestand Zorgverzekeringswet 2024 - gemeente.csv", sep=";")

In [ ]:
# First look at the top rows, including the unlabelled bucket row (row 0)
vektis.head()

## Step 1. Privacy bucket

In [ ]:
# Privacy bucket: groups under 10 people (hidden so no one can be identified)
# plus people with unknown sex, age or municipality. All lumped into one unlabelled row.
# Find it: the only row with no municipality name
bucket = vektis[vektis['gemeentenaam'].isna()]
bucket

In [ ]:
# Size in people: the bucket's share of everyone insured
bucket_people = bucket["aantal_bsn"].sum()
total_people = vektis["aantal_bsn"].sum()
share = bucket_people / total_people * 100
print(f"Bucket: {bucket_people:,.0f} people, {share:.1f}% of all people")

In [ ]:
# All 26 cost columns, and their grand total (bucket still included at this point)
costs = vektis.filter(like="kosten")
total_costs = costs.sum().sum()

In [ ]:
# Size in euros: the bucket's share of all cost
bucket_cost = bucket.filter(like="kosten").sum().sum()
cost_share = bucket_cost / total_costs * 100
print(f"Bucket: €{bucket_cost:,.0f}, {cost_share:.1f}% of all cost")

**Decision: privacy bucket**
- Found: one row with empty sex, age and municipality. Vektis puts groups under 10 people (privacy) and people with unknown sex, age or municipality here. It holds 412,394 people (2.2%) but only €167.8 million (0.3% of all cost).
- Did: removed the row. The raw file stays unchanged.
- Why: every KPI needs a municipality, and expected cost needs age and sex. This row has none of them, so it cannot be placed anywhere. Rejected: spreading it over municipalities, because we do not know where these people live.
- Effect: only 5 small cells are missing (Rozendaal, Schiermonnikoog, Vlieland). The bucket is cheap per person (about €760 per insured-year against €3,382), likely because many were insured for only part of the year. Removing it barely changes the national figures.

In [ ]:
# Remove the bucket: keep only rows that have a municipality name
vektis = vektis[vektis["gemeentenaam"].notna()]

In [ ]:
# Check the size: 12,991 rows expected
vektis.shape

In [ ]:
# Stop with an error if the row count is not 12,991
assert len(vektis) == 12991

## Step 2. Negative costs

In [ ]:
# Recreate after dropping the bucket, so costs and vektis have the same rows
costs = vektis.filter(like="kosten")
total_costs = costs.sum().sum()

In [ ]:
# How many negative cells?
neg_costs = costs < 0
neg_costs.sum().sum()

In [ ]:
# Which rows are they, and how big?
neg_rows = vektis[neg_costs.any(axis=1)]
neg_rows[["geslacht", "leeftijdsklasse", "gemeentenaam",
          "kosten_integrale_geboortezorg", "kosten_intramuraal_verblijf_ggz"]]

In [ ]:
# Total cost per row (26 columns added across), then per municipality.
# Compare the two negative cells with their municipality's total.
row_total = costs.sum(axis=1)
town_total = row_total.groupby(vektis["gemeentenaam"]).sum()
town_total[["UITGEEST", "TWENTERAND"]]

**Decision: negative costs**
- Found: 2 negative cells out of about 338,000: Uitgeest, women 25-29, inpatient mental health stays (−€2,111), and Twenterand, women 30-34, birth care (−€2,300). Each is under 0.01% of that municipality's total cost.
- Did: kept them as they are.
- Why: they are real corrections (reversed or refunded claims), too small to move any result, and keeping them means totals match the official Vektis figures. Rejected: setting them to zero (totals would no longer match Vektis) and dropping the rows (would also remove 25 real cost cells per row).

## Step 3. Care abroad

In [ ]:
# Care abroad: total euros, and its share of all cost
abroad = vektis["kosten_grensoverschrijdende_zorg"]
abroad_costs = abroad.sum()
abroad_share = abroad_costs / total_costs * 100
print(f"Care abroad: €{abroad_costs:,.0f}, {abroad_share:.1f}% of all cost")

In [ ]:
# Care abroad as a share of each municipality's own cost, largest first
abroad_town = vektis["kosten_grensoverschrijdende_zorg"].groupby(vektis["gemeentenaam"]).sum()
abroad_pct = abroad_town / town_total * 100
abroad_pct.sort_values(ascending=False)

In [ ]:
# The typical municipality, to compare the top ones against
abroad_pct.median()

**Decision: care abroad**
- Found: care abroad (`kosten_grensoverschrijdende_zorg`) is €347.3 million, 0.6% of all cost, and the leaflet says it is incomplete because foreign bills arrive late. It is concentrated in border municipalities: median 0.4% of a municipality's cost, but Sluis 18%, Hulst 10%, Terneuzen 8% (Zeeuws-Vlaanderen, next to Belgium) and Vaals 5% (next to Germany and Belgium).
- Did: kept the column.
- Why: for border towns, care abroad is part of their normal hospital care. Excluding it would make Sluis look about 18% cheaper than it is and give the purchasing team a false signal. Rejected: excluding the column.
- Limitation: because the column is incomplete, border towns' true cost is probably higher than shown. This goes in the README's Limitations.

## Step 4. Age band as a number

In [ ]:
# The problem: sorted as text, '5 t/m 9' lands after '45 t/m 49'
sorted(vektis["leeftijdsklasse"].unique())

In [ ]:
# New column: the first number in each label (the lower age), as a real number
vektis["age_from"] = vektis["leeftijdsklasse"].str.extract(r"(\d+)").astype(int)

In [ ]:
# Check: each label once, ordered by the new number (expect 19 rows, 0 to 90)
vektis[["leeftijdsklasse", "age_from"]].drop_duplicates().sort_values("age_from")

**Note: age order.** Age bands are text, so they sort wrongly ("5 t/m 9" after "45 t/m 49"). Added `age_from`, the lower age as a number (0, 5 ... 90), for ordering and charts. The text label stays for display.

## Step 5. Care-type groups

In [ ]:
# Load the mapping: which of the 10 groups each of the 26 cost columns belongs to
groups = pd.read_csv("../data/manual/care_type_groups.csv")